![Banner](banner.jpg)

# Laboratorio 8: Redes Neuronales Convolucionales (CNNs)

## 1. Introducción

En este laboratorio vamos a explorar cómo construir una **Red Neuronal Convolucional (CNN)** usando TensorFlow/Keras para clasificar imágenes reales vs generadas por inteligencia artificial.

### ¿Qué es una CNN?

Una Red Neuronal Convolucional (CNN) es un tipo de red neuronal profunda diseñada específicamente para procesar datos como las imágenes.
A diferencia de las redes densas (MLP), las CNNs aplican filtros (kernels) que se deslizan sobre la imagen para detectar patrones locales como bordes, texturas y formas.
Estos filtros se aprenden automáticamente durante el entrenamiento, permitiendo que la red extraiga características cada vez más complejas en capas más profundas.

### Objetivos del laboratorio

1. Cargar y explorar el dataset de imágenes reales vs generadas por IA
2. Comprender los conceptos de convoluciones, kernels y pooling
3. Construir una CNN desde cero con TensorFlow/Keras
4. Entrenar y evaluar el modelo
5. Aplicar Data Augmentation para mejorar la generalización

## 2. Preparación del entorno

Instalamos las dependencias necesarias e importamos las librerías.

In [ ]:
!pip install tensorflow matplotlib numpy kagglehub scikit-learn

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path
import random

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 3. Carga y exploración del dataset

Utilizaremos el dataset **AI vs Real Images** de Kaggle, que contiene ~1500 imágenes en 2 clases (real y generada por IA) distribuidas en 5 categorías: Naturaleza, Ciudad, Animales, Comida y Personas.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download('rhythmghai/ai-vs-real-images-dataset')
print(f'Dataset descargado en: {dataset_path}')

In [ ]:
# Explorar la estructura de carpetas
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    folder_name = os.path.basename(root)
    num_files = len([f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))])
    if num_files > 0:
        print(f'{indent}{folder_name}/ ({num_files} imágenes)')
    else:
        print(f'{indent}{folder_name}/')

In [ ]:
# Reorganizar datos en estructura binaria para usar image_dataset_from_directory
import shutil
from pathlib import Path

organized_dir = Path(dataset_path) / 'organized'
for class_name, source_folder in [('real', 'real_dataset'), ('ai_generated', 'ai_generated_dataset')]:
    class_dir = organized_dir / class_name
    class_dir.mkdir(parents=True, exist_ok=True)
    source_path = Path(dataset_path) / source_folder
    if source_path.exists():
        for img_path in source_path.rglob('*'):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                dest = class_dir / f'{img_path.parent.name}_{img_path.name}'
                if not dest.exists():
                    shutil.copy2(img_path, dest)

# Contar imágenes por clase
for class_dir in sorted(organized_dir.iterdir()):
    count = len(list(class_dir.glob('*')))
    print(f'{class_dir.name}: {count} imágenes')

In [ ]:
# Visualizar imágenes de ejemplo (4 reales, 4 generadas por IA)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for row, class_name in enumerate(['real', 'ai_generated']):
    class_dir = organized_dir / class_name
    image_files = sorted(class_dir.glob('*'))[:4]
    for col, img_path in enumerate(image_files):
        img = tf.keras.utils.load_img(img_path, target_size=(150, 150))
        axes[row, col].imshow(img)
        axes[row, col].set_title(class_name, fontsize=10)
        axes[row, col].axis('off')

plt.suptitle('Muestras del dataset: Real vs IA', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Preprocesamiento de los datos

Redimensionamos todas las imágenes a 150×150 píxeles, normalizamos los valores de píxeles al rango [0, 1] y dividimos el dataset en conjuntos de entrenamiento, validación y prueba.

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32

full_dataset = tf.keras.utils.image_dataset_from_directory(
    organized_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    seed=42,
    label_mode='binary'
)
class_names = full_dataset.class_names
print(f'Clases: {class_names}')

In [ ]:
# Dividir en train (70%), validación (15%) y test (15%)
dataset_size = full_dataset.cardinality().numpy()
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)

train_ds = full_dataset.take(train_size)
remaining = full_dataset.skip(train_size)
val_ds = remaining.take(val_size)
test_ds = remaining.skip(val_size)

print(f'Batches - Train: {train_ds.cardinality().numpy()}, '
      f'Val: {val_ds.cardinality().numpy()}, '
      f'Test: {test_ds.cardinality().numpy()}')

In [ ]:
# Normalizar píxeles al rango [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
# Optimizar rendimiento con caché y prefetch
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 5. Conceptos fundamentales de CNNs

### 5.1 Convoluciones y Kernels

Una **convolución** es una operación que aplica un filtro (kernel) sobre una imagen para extraer características.
El kernel es una pequeña matriz de pesos que se desliza sobre la imagen. En cada posición, se calcula el producto punto entre el kernel y la región de la imagen.
Diferentes kernels detectan diferentes características: bordes horizontales, bordes verticales, texturas, patrones, etc.
En una CNN, los valores del kernel **se aprenden automáticamente** durante el entrenamiento mediante backpropagation.

In [ ]:
# Tomar una imagen del dataset
for images, labels in train_ds.take(1):
    sample_image = images[0].numpy()
    break

# Convertir a escala de grises
gray_image = tf.image.rgb_to_grayscale(sample_image)
gray_image = tf.squeeze(gray_image)

# Definir kernels de detección de bordes
kernel_horizontal = tf.constant([[-1, -1, -1],
                                  [ 0,  0,  0],
                                  [ 1,  1,  1]], dtype=tf.float32)

kernel_vertical = tf.constant([[-1, 0, 1],
                                [-1, 0, 1],
                                [-1, 0, 1]], dtype=tf.float32)

# Reshape para tf.nn.conv2d: [batch, height, width, channels]
image_4d = tf.reshape(gray_image, [1, gray_image.shape[0], gray_image.shape[1], 1])
kernel_h = tf.reshape(kernel_horizontal, [3, 3, 1, 1])
kernel_v = tf.reshape(kernel_vertical, [3, 3, 1, 1])

# Aplicar convoluciones
edges_h = tf.nn.conv2d(image_4d, kernel_h, strides=[1,1,1,1], padding='SAME')
edges_v = tf.nn.conv2d(image_4d, kernel_v, strides=[1,1,1,1], padding='SAME')

# Visualizar
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(sample_image)
axes[0].set_title('Imagen original')
axes[1].imshow(tf.squeeze(gray_image), cmap='gray')
axes[1].set_title('Escala de grises')
axes[2].imshow(tf.squeeze(edges_h), cmap='gray')
axes[2].set_title('Bordes horizontales')
axes[3].imshow(tf.squeeze(edges_v), cmap='gray')
axes[3].set_title('Bordes verticales')
for ax in axes:
    ax.axis('off')
plt.suptitle('Efecto de diferentes kernels de convolución', fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 Pooling

**Pooling** reduce las dimensiones espaciales de los feature maps, disminuyendo el costo computacional.
- **Max Pooling**: toma el valor máximo en cada ventana → conserva las características más prominentes.
- **Average Pooling**: toma el promedio en cada ventana → suaviza las activaciones.
- **Beneficios**: reduce parámetros, previene overfitting y proporciona invariancia a pequeñas traslaciones.

In [ ]:
# Ejemplo numérico de pooling
example = tf.constant([[1, 2, 3, 4],
                        [5, 6, 7, 8],
                        [9, 10, 11, 12],
                        [13, 14, 15, 16]], dtype=tf.float32)
example_4d = tf.reshape(example, [1, 4, 4, 1])

max_pool = tf.keras.layers.MaxPooling2D(pool_size=2)(example_4d)
avg_pool = tf.keras.layers.AveragePooling2D(pool_size=2)(example_4d)

print('Matriz original (4x4):')
print(example.numpy())
print(f'\nMax Pooling 2x2 → ({max_pool.shape[1]}x{max_pool.shape[2]}):')
print(tf.squeeze(max_pool).numpy())
print(f'\nAverage Pooling 2x2 → ({avg_pool.shape[1]}x{avg_pool.shape[2]}):')
print(tf.squeeze(avg_pool).numpy())

In [ ]:
# Aplicar pooling a una imagen real
image_for_pool = tf.reshape(gray_image, [1, gray_image.shape[0], gray_image.shape[1], 1])
max_pooled = tf.keras.layers.MaxPooling2D(pool_size=4)(image_for_pool)
avg_pooled = tf.keras.layers.AveragePooling2D(pool_size=4)(image_for_pool)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(tf.squeeze(gray_image), cmap='gray')
axes[0].set_title(f'Original ({gray_image.shape[0]}x{gray_image.shape[1]})')
axes[1].imshow(tf.squeeze(max_pooled), cmap='gray')
axes[1].set_title(f'Max Pooling ({max_pooled.shape[1]}x{max_pooled.shape[2]})')
axes[2].imshow(tf.squeeze(avg_pooled), cmap='gray')
axes[2].set_title(f'Avg Pooling ({avg_pooled.shape[1]}x{avg_pooled.shape[2]})')
for ax in axes:
    ax.axis('off')
plt.suptitle('Efecto del pooling en una imagen real', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Arquitectura de una CNN

La arquitectura típica de una CNN sigue el siguiente pipeline:

**Entrada → [Conv2D → ReLU → MaxPool] × N → Flatten → Dense → Salida**

- **Conv2D** extrae características locales de la imagen mediante filtros aprendidos.
- **MaxPool** reduce las dimensiones espaciales, manteniendo las características más relevantes.
- Se apilan múltiples bloques convolucionales para detectar patrones cada vez más complejos (bordes → texturas → objetos).
- **Flatten** convierte el tensor 3D resultante en un vector 1D para alimentar las capas densas.
- Las capas **Dense** realizan la clasificación final basándose en las características extraídas.

## 6. Construcción del modelo CNN

Construiremos una CNN con 3 bloques convolucionales (Conv2D + MaxPooling) seguidos de un clasificador denso con Dropout para regularización.

In [ ]:
model = tf.keras.Sequential([
    # Bloque 1
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 2
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 3
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Clasificador
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
# Contar parámetros del modelo
total_params = model.count_params()
print(f'\nTotal de parámetros: {total_params:,}')
print(f'Parámetros entrenables: {sum(p.numpy().size for p in model.trainable_weights):,}')

## 7. Entrenamiento

Compilamos el modelo con el optimizador Adam y la función de pérdida binary crossentropy, adecuada para clasificación binaria.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
# Graficar curvas de aprendizaje
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['loss'], label='Entrenamiento')
ax1.plot(history.history['val_loss'], label='Validación')
ax1.set_title('Pérdida (Loss)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], label='Entrenamiento')
ax2.plot(history.history['val_accuracy'], label='Validación')
ax2.set_title('Precisión (Accuracy)')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Curvas de aprendizaje - CNN sin aumentación', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Evaluación

Evaluamos el modelo en el conjunto de prueba y analizamos los resultados con una matriz de confusión.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f'\nPérdida en test: {test_loss:.4f}')
print(f'Precisión en test: {test_accuracy:.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import numpy as np

y_true = []
y_pred = []
for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred.extend((predictions > 0.5).astype(int).flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues')
plt.title('Matriz de Confusión - CNN sin aumentación')
plt.show()

print('\nReporte de clasificación:')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Visualizar predicciones individuales
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for images, labels in test_ds.take(1):
    predictions = model.predict(images, verbose=0)
    for i, ax in enumerate(axes.flatten()):
        if i < len(images):
            ax.imshow(images[i].numpy())
            pred_class = class_names[int(predictions[i] > 0.5)]
            true_class = class_names[int(labels[i])]
            color = 'green' if pred_class == true_class else 'red'
            ax.set_title(f'Pred: {pred_class}\nReal: {true_class}', color=color, fontsize=10)
            ax.axis('off')
plt.suptitle('Predicciones del modelo', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Aumentación de datos (Data Augmentation)

**Data Augmentation** consiste en aplicar transformaciones aleatorias a las imágenes de entrenamiento.
Esto genera más variedad sin necesidad de recopilar nuevos datos y ayuda a prevenir el sobreajuste (overfitting).
Transformaciones comunes incluyen: rotación, volteo horizontal/vertical, zoom y traslación.
En Keras, se pueden integrar directamente como capas del modelo usando `tf.keras.layers`.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

# Visualizar el efecto de la aumentación
for images, _ in train_ds.take(1):
    sample = images[0]
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes[0, 0].imshow(sample.numpy())
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    for i, ax in enumerate(axes.flatten()[1:], 1):
        augmented = data_augmentation(tf.expand_dims(sample, 0))
        ax.imshow(augmented[0].numpy())
        ax.set_title(f'Aumentada {i}')
        ax.axis('off')
    plt.suptitle('Ejemplos de Data Augmentation', fontsize=14)
    plt.tight_layout()
    plt.show()
    break

In [ ]:
# Modelo con Data Augmentation integrada
model_aug = tf.keras.Sequential([
    # Capas de aumentación (solo se aplican durante entrenamiento)
    tf.keras.layers.RandomFlip('horizontal', input_shape=(150, 150, 3)),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),

    # Bloque 1
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 2
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Bloque 3
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),

    # Clasificador
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_aug.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_aug.summary()

In [ ]:
history_aug = model_aug.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15
)

In [ ]:
# Comparar curvas de aprendizaje de ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Sin aug. (train)', linestyle='--')
axes[0].plot(history.history['val_loss'], label='Sin aug. (val)', linestyle='--')
axes[0].plot(history_aug.history['loss'], label='Con aug. (train)')
axes[0].plot(history_aug.history['val_loss'], label='Con aug. (val)')
axes[0].set_title('Comparación de Pérdida')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Sin aug. (train)', linestyle='--')
axes[1].plot(history.history['val_accuracy'], label='Sin aug. (val)', linestyle='--')
axes[1].plot(history_aug.history['accuracy'], label='Con aug. (train)')
axes[1].plot(history_aug.history['val_accuracy'], label='Con aug. (val)')
axes[1].set_title('Comparación de Precisión')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Comparación: CNN sin vs con Data Augmentation', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluar modelo con aumentación en test
test_loss_aug, test_acc_aug = model_aug.evaluate(test_ds)
print(f'\n--- Comparación de resultados en Test ---')
print(f'Sin aumentación: Accuracy = {test_accuracy:.4f}, Loss = {test_loss:.4f}')
print(f'Con aumentación: Accuracy = {test_acc_aug:.4f}, Loss = {test_loss_aug:.4f}')